In [1]:
import pandas as pd

In [2]:
# Test queries for RAG evaluation
# Organized by difficulty and emotional coverage

TEST_QUERIES = [
    # ═══════════════════════════════════════════════════════
    # EASY: Direct keyword/topic matches
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q001",
        "query": "I need help setting boundaries with my family",
        "expected_episodes": ["001"],  # Mark Manson boundaries
        "expected_themes": ["boundaries", "saying no", "relationships"],
        "difficulty": "easy",
        "emotion": "frustrated",
    },
    {
        "id": "Q002",
        "query": "I keep comparing myself to people on social media and feeling inadequate",
        "expected_episodes": ["017"],  # Comparing yourself to others
        "expected_themes": ["comparison", "social media", "self-worth"],
        "difficulty": "easy",
        "emotion": "inadequate",
    },
    {
        "id": "Q003",
        "query": "I'm scared of being judged and it's blocking my creativity",
        "expected_episodes": ["010"],  # Fear of being judged
        "expected_themes": ["judgment", "creativity", "fear"],
        "difficulty": "easy",
        "emotion": "fear",
    },
    
    # ═══════════════════════════════════════════════════════
    # MEDIUM: Semantic understanding required
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q004",
        "query": "I keep beating myself up over mistakes I made at work",
        "expected_episodes": ["002", "003"],  # Self-compassion + Trusting yourself
        "expected_themes": ["self-compassion", "shame", "mistakes", "self-criticism"],
        "difficulty": "medium",
        "emotion": "shame",
    },
    {
        "id": "Q005",
        "query": "Why do I sabotage good relationships when things are going well?",
        "expected_episodes": ["004", "033"],  # Self-sabotage + Self-abandonment
        "expected_themes": ["self-sabotage", "relationships", "patterns"],
        "difficulty": "medium",
        "emotion": "confused",
    },
    {
        "id": "Q006",
        "query": "I feel stuck in life and don't know what direction to take",
        "expected_episodes": ["011", "026"],  # Life feels empty + feeling lost
        "expected_themes": ["lost", "direction", "purpose", "uncertainty"],
        "difficulty": "medium",
        "emotion": "lost",
    },
    {
        "id": "Q007",
        "query": "I'm constantly anxious and my mind won't stop racing",
        "expected_episodes": ["019", "020", "036", "039"],  # Stopping anxiety (multiple)
        "expected_themes": ["anxiety", "overthinking", "calm", "nervous system"],
        "difficulty": "medium",
        "emotion": "anxious",
    },
    {
        "id": "Q008",
        "query": "I feel lonely even though I'm surrounded by people",
        "expected_episodes": ["018"],  # Simon Sinek on loneliness
        "expected_themes": ["loneliness", "connection", "isolation"],
        "difficulty": "medium",
        "emotion": "lonely",
    },
    
    # ═══════════════════════════════════════════════════════
    # HARD: Requires inference and emotional understanding
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q009",
        "query": "People always take advantage of me and I can't say no",
        "expected_episodes": ["001", "033"],  # Boundaries + people-pleasing
        "expected_themes": ["boundaries", "people-pleasing", "self-worth"],
        "difficulty": "hard",
        "emotion": "resentful",
    },
    {
        "id": "Q010",
        "query": "I don't feel good enough no matter what I achieve",
        "expected_episodes": ["008", "034", "035"],  # Perfectionism + self-love
        "expected_themes": ["perfectionism", "self-worth", "achievement", "enough"],
        "difficulty": "hard",
        "emotion": "inadequate",
    },
    {
        "id": "Q011",
        "query": "I'm grieving a breakup and can't stop thinking about what I did wrong",
        "expected_episodes": ["025", "028"],  # Grief + marriage/relationships
        "expected_themes": ["grief", "heartbreak", "relationships", "self-blame"],
        "difficulty": "hard",
        "emotion": "heartbroken",
    },
    {
        "id": "Q012",
        "query": "I want to make a change but I'm terrified of failing",
        "expected_episodes": ["009", "016", "023"],  # Transform life + fear + overcome fear
        "expected_themes": ["change", "fear", "failure", "courage"],
        "difficulty": "hard",
        "emotion": "afraid",
    },
    
    # ═══════════════════════════════════════════════════════
    # EDGE CASES: Test system limits
    # ═══════════════════════════════════════════════════════
    {
        "id": "Q013",
        "query": "I'm tired all the time and feel unmotivated to do anything",
        "expected_episodes": ["026", "027", "030"],  # Lost/lazy/unmotivated + trauma
        "expected_themes": ["burnout", "fatigue", "motivation", "trauma"],
        "difficulty": "hard",
        "emotion": "exhausted",
        "note": "Tests if system connects fatigue to deeper emotional/trauma issues"
    },
    {
        "id": "Q014",
        "query": "I need to build better habits but keep falling back into old patterns",
        "expected_episodes": ["007", "029"],  # Negative patterns + habits
        "expected_themes": ["habits", "patterns", "change", "consistency"],
        "difficulty": "medium",
        "emotion": "frustrated",
    },
    {
        "id": "Q015",
        "query": "I want to love myself but I don't know where to start",
        "expected_episodes": ["034", "035"],  # How to love yourself (2 episodes)
        "expected_themes": ["self-love", "self-worth", "self-acceptance"],
        "difficulty": "easy",
        "emotion": "confused",
        "note": "Tests if duplicate episodes are handled (ep 034 and 035 both on self-love)"
    },
]

In [ ]:
# notebooks/evaluation.ipynb

import sys
import os

sys.path.append(os.path.abspath(".."))

from scripts.rag_pipeline import run_pipeline
from src.vector_store import get_collection
from src.memory import ConversationMemory

collection = get_collection()
memory = ConversationMemory()

results = []

for test_case in TEST_QUERIES:
    print(f"\n{'='*60}")
    print(f"Query {test_case['id']}: {test_case['query']}")
    print(f"Expected: {test_case['expected_episodes']}")
    print(f"Difficulty: {test_case['difficulty']}")
    
    # Run pipeline
    output = run_pipeline(
        user_query=test_case['query'],
        collection=collection,
        memory=memory,
        top_k=5,
    )
    
    # Extract top episode IDs
    retrieved = [r['metadata']['episode_id'] for r in output['recommendations'][:5]]
    
    # Calculate Precision@3
    relevant_in_top3 = len(set(retrieved[:3]) & set(test_case['expected_episodes']))
    precision_3 = relevant_in_top3 / 3
    
    # Calculate Recall@5
    recall_5 = len(set(retrieved) & set(test_case['expected_episodes'])) / len(test_case['expected_episodes'])
    
    print(f"Retrieved: {retrieved[:3]}")
    print(f"Precision@3: {precision_3:.2f}")
    print(f"Recall@5: {recall_5:.2f}")
    
    results.append({
        'query_id': test_case['id'],
        'query': test_case['query'],
        'difficulty': test_case['difficulty'],
        'emotion': test_case['emotion'],
        'precision_3': precision_3,
        'recall_5': recall_5,
        'retrieved_top3': retrieved[:3],
        'expected': test_case['expected_episodes'],
    })

# Summary
df_results = pd.DataFrame(results)
print("\n" + "="*60)
print("OVERALL RESULTS")
print("="*60)
print(f"Average Precision@3: {df_results['precision_3'].mean():.3f}")
print(f"Average Recall@5: {df_results['recall_5'].mean():.3f}")
print(f"\nBy Difficulty:")
print(df_results.groupby('difficulty')['precision_3'].mean())

create experiment object to log the evaluation metrics 

In [5]:
import json
from datetime import datetime

experiment = {
    "experiment_id": f"exp_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "retriever": "embedding_only",
    "embedding_model": "text-embedding-3-small",
    "dataset_size": collection.count(),
    "queries": len(TEST_QUERIES),
    "precision@3": round(df_results["precision_3"].mean(), 3),
    "recall@5": round(df_results["recall_5"].mean(), 3),
    "timestamp": datetime.now().isoformat()
}

logging in json

In [6]:
log_path = "../data/evaluation/retrieval_accuracy.json"

# Load existing experiments
try:
    with open(log_path, "r") as f:
        experiments = json.load(f)
except:
    experiments = []

# Append new experiment
experiments.append(experiment)

# Save updated log
with open(log_path, "w") as f:
    json.dump(experiments, f, indent=2)

print("Experiment logged successfully")

Experiment logged successfully
